In [1]:
using Pkg

Pkg.activate("mnist")
#Pkg.add("MLDatasets")
#Pkg.add("Images")
Pkg.add("DataFrames")
using MLDatasets
using Images

  Activating project at `~/Desktop/DataAssim.jl/mnist`
   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/mnist/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/mnist/Manifest.toml`


In [2]:
Pkg.develop(path="../Krylov.jl")   # change path to local Krylov fork
Pkg.develop(path="../JSOSolvers.jl") 

   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/mnist/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/mnist/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/mnist/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/mnist/Manifest.toml`


In [3]:
F = MLDatasets.MNIST(split = :train)
A = F.features  # images d'entrée
b = F.targets   # labels
println("$(length(b)) images")

60000 images


In [ ]:
using Random

# extraction des parties de A et b pertinentes
digits = (1, 7)
index_digits = findall(x -> x ∈ digits, b)

#n_samples = 6000  # taille  souhaitée
#idx_sub = index_digits[randperm(length(index_digits))[1:n_samples]]
b_digits = b[index_digits]
A_digits = A[:, :, index_digits]

# définition des deux classes
b_digits[b_digits .== digits[1]] .= 1
b_digits[b_digits .== digits[2]] .= -1

# reformulation de A sous forme d'une matrice
# chaque colonne du nouveau A_digits est la vectorisation d'une des images,
# i.e., l'empilement de ses colonnes les unes par-dessus les autres
A_digits = reshape(A_digits, size(A_digits, 1) * size(A_digits, 2), size(A_digits, 3))
A_digits = convert(Matrix{Float64}, A_digits) ./ 255

size(A_digits)

(784, 6000)

In [5]:
using NLPModels, ADNLPModels
using LinearAlgebra
using SparseArrays

In [6]:

Ahat = Diagonal(b_digits) * sparse(A_digits)'
println(size(Ahat))
function f(x)
    r = Ahat * x               # Ahat peut être sparse ou dense
    r .= 100.0 .- 100.0 .* tanh.(r)
    return 0.5 * dot(r, r)
end
   
x0 = ones(size(A_digits, 1))
nlp = ADNLPModel(f, x0)


(6000, 784)


ADNLPModel - Model with automatic differentiation backend ADModelBackend{
  ForwardDiffADGradient,
  ForwardDiffADHvprod,
  EmptyADbackend,
  EmptyADbackend,
  EmptyADbackend,
  SparseADHessian,
  EmptyADbackend,
}
  Problem name: Generic
   All variables: ████████████████████ 784    All constraints: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
            free: ████████████████████ 784               free: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           lower: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                lower: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           upper: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                upper: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
         low/upp: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0              low/upp: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           fixed: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                fixed: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
          infeas: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0               infeas: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
            nnzh: (  0.00% sparsity)   307720          linear: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
                                 

In [7]:
using JSOSolvers

In [8]:
using DataFrames
function run_solver(nlp, subsolver; memory=nothing)
    kwargs = memory === nothing ? () : (subsolver_kwargs=(memory=memory,),)

    stats = trunk(nlp,
        max_time=10000.0,
        max_iter=500,
        verbose=0,
        subsolver=subsolver;
        kwargs...
    )

    row = (
        solver = string(subsolver) * (memory === nothing ? "" : "_m$(memory)"),
        status = stats.status,
        norm_sol = norm(stats.solution),
        objective = stats.objective,
        iter = stats.iter,
        obj_eval = nlp.counters.neval_obj,
        grad_eval = nlp.counters.neval_grad,
        hprod = nlp.counters.neval_hprod,
        time = stats.elapsed_time
    )

    reset!(nlp)

    return row
end

run_solver (generic function with 1 method)

In [9]:
results = DataFrame()

push!(results, run_solver(nlp, :cg))
push!(results, run_solver(nlp, :lbfgs, memory=100))
push!(results, run_solver(nlp, :diom, memory=100))
push!(results, run_solver(nlp, :lbfgs, memory=50))
push!(results, run_solver(nlp, :diom, memory=50))

Row,solver,status,norm_sol,objective,iter,obj_eval,grad_eval,hprod,time
,String,Symbol,Float64,Float64,Int64,Int64,Int64,Int64,Float64
1,cg,first_order,5342.4,40000.3,20,21,21,263,714.366
2,lbfgs_m100,first_order,5342.07,40000.3,20,21,21,265,732.018
3,diom_m100,first_order,5341.92,40000.3,20,21,21,250,676.156
4,lbfgs_m50,first_order,5342.07,40000.3,20,21,21,265,714.862
5,diom_m50,first_order,5341.92,40000.3,20,21,21,250,687.184


In [10]:
using PrettyTables
pretty_table(results)

┌────────────┬─────────────┬──────────┬───────────┬───────┬──────────┬──────────
│     solver │      status │ norm_sol │ objective │  iter │ obj_eval │ grad_ev ⋯
│     String │      Symbol │  Float64 │   Float64 │ Int64 │    Int64 │     Int ⋯
├────────────┼─────────────┼──────────┼───────────┼───────┼──────────┼──────────
│         cg │ first_order │   5342.4 │   40000.3 │    20 │       21 │         ⋯
│ lbfgs_m100 │ first_order │  5342.07 │   40000.3 │    20 │       21 │         ⋯
│  diom_m100 │ first_order │  5341.92 │   40000.3 │    20 │       21 │         ⋯
│  lbfgs_m50 │ first_order │  5342.07 │   40000.3 │    20 │       21 │         ⋯
│   diom_m50 │ first_order │  5341.92 │   40000.3 │    20 │       21 │         ⋯
└────────────┴─────────────┴──────────┴───────────┴───────┴──────────┴──────────
                                                               3 columns omitted
